# Vélib Station Availability - Raw Data Ingestion

### Objective
This notebook retrieves  records from the real-time Vélib station availability API and lands the untouched JSON response directly into a Unity Catalog Volume.

### Data Flow
Paris OpenData API  → Raw JSON Payload → Databricks UC Volume

### Source
Paris OpenData Dataset: *Vélib - Vélos et bornes - Disponibilité temps réel*

### Storage Location
/Volumes/workspace/default/raw_data/velib_disponibilite_records_raw.json

### Processing
No transformations or schema enforcement are applied at this stage. The goal is pure raw data preservation (Landing Zone pattern).

In [0]:
import os
import requests
import json

# 1. Base API URL & Volume Path Setup
BASE_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/records"
VOLUME_PATH = "/Volumes/workspace/default/raw_data"
FILE_NAME = "velib_disponibilite_raw.json"
TARGET_FILE_PATH = os.path.join(VOLUME_PATH, FILE_NAME)

os.makedirs(VOLUME_PATH, exist_ok=True)

# 2. Loop & Paginate
all_records = []
limit = 100
offset = 0

print("Fetching all records via API pagination...")

while True:
    params = {"limit": limit, "offset": offset}
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    
    data = response.json()
    results = data.get("results", [])
    
    if not results:
        break  # Stop when no more records are returned
        
    all_records.extend(results)
    offset += limit
    print(f"Fetched {len(all_records)} / {data.get('total_count', 'unknown')} records...")

# 3. Write Complete JSON Array to Volume
with open(TARGET_FILE_PATH, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print(f"\nSuccessfully saved {len(all_records)} total records to {TARGET_FILE_PATH}!")

In [0]:
# 1. Read the complete JSON array directly into Spark
df = spark.read.option("multiline", "true").json(
    "/Volumes/workspace/default/raw_data/velib_disponibilite_raw.json"
)

# 2. Check total row count (Will output ~1500+)
print(f"Total Rows Ingested: {df.count()}")

# 3. Reorder columns to match the original Paris OpenData dataset
original_column_order = [
    "stationcode",
    "name",
    "is_installed",
    "capacity",
    "numdocksavailable",
    "numbikesavailable",
    "mechanical",
    "ebike",
    "is_renting",
    "is_returning",
    "duedate",
    "coordonnees_geo",
    "nom_arrondissement_communes",
    "code_insee_commune",
    "station_opening_hours"
]


# Keep only columns that actually exist in the DataFrame
original_column_order = [
    col for col in original_column_order if col in df.columns
]

df = df.select(original_column_order)

# Display with the correct column order
display(df)